# RSNA Knee Abnormality Detection — Phase 4C.7 gold-only constrained evaluation

Internet-off Kaggle notebook. **Do not enable internet. GPU is required.**
This notebook scores the **58 gold studies only** with transport **v1.4** and
the corrected constrained generator. It does **not** process the 4,349
report-only studies, does **not** train, does **not** create folds, and does
**not** submit.

Create a **new** kernel slug `rsna-knee-second-reader-gold-constrained-v14`.
Do **not** overwrite gold kernel v7, constrained-canary v3, canary v1, smoke
v2, or smoke/canary v1.3. Do **not** execute notebook 07.

Prompt remains **v1.2.0**. Schema remains **v1.0.0**. Frozen v1.4 YAML still
records `prefix_allowed_tokens_fn: true`; effective runtime does **not** pass
that function to `model.generate`. Source code is authoritative. Exactly one
schema-mask logits processor. Phi EOS IDs `[32000, 32007, 32001]`.

Acceptance uses the frozen gold criteria (positive precision, NPV, CI lower
bounds, malformed rate, small-n cannot PILOT_PASS). Coverage is reported, not
a pass gate. Primary system: strict agreement.
Never record `CONSTRAINED_FORMAT_PASS_SEMANTIC_PASS`.
`gold_rerun_authorized=false` after this run. `HARD_WEAK_LABEL_TRAINING_ALLOWED=false`.
`SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED=false`.


In [ ]:
# Bootstrap: stdlib only until the package is importable.
# Copies/extracts source under /kaggle/working. Never writes under /kaggle/input.
from __future__ import annotations

import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_second_reader_gold_constrained_v14"
MARKERS = ("knee/config.py", "knee/second_reader_phase47.py")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks", "vendor", "third_party"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md", "literature/ontology.yaml"):
        src = src_root / filename
        if src.is_file():
            dest_file = dest / filename
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest_file)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


SKIP_WALK_DIRS = {"train_series", "test_series", "__pycache__", ".git", "competitions"}


def _walk_input_for_source(root: Path, max_depth: int = 6):
    zips: list[Path] = []
    dirs: list[Path] = []
    try:
        root_res = root.resolve()
    except OSError:
        return zips, dirs
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        try:
            rel = current.resolve().relative_to(root_res)
            depth = 0 if str(rel) == "." else len(rel.parts)
        except (OSError, ValueError):
            dirnames[:] = []
            continue
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP_WALK_DIRS and d != COMPETITION_SLUG)
        if depth > max_depth:
            dirnames[:] = []
            continue
        if _looks_like_source(current):
            dirs.append(current)
        for name in filenames:
            if name.lower().endswith(".zip"):
                zip_path = current / name
                if _zip_has_source(zip_path):
                    zips.append(zip_path)
    return zips, dirs


def locate_and_extract_source() -> Path | None:
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips: list[Path] = []
    dirs: list[Path] = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and child.suffix.lower() == ".zip" and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            found_zips, found_dirs = _walk_input_for_source(child)
            zips.extend(found_zips)
            dirs.extend(found_dirs)
    if zips:
        print(f"extracting_source_zip={zips[0]}")
        _safe_extract(zips[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        for nested in sorted(p for p in SRC_DST.iterdir() if p.is_dir()):
            if _looks_like_source(nested):
                return nested
    if dirs:
        print(f"copying_source_tree={dirs[0]}")
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
if SOURCE_ROOT is not None:
    resolved = str(SOURCE_ROOT.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"source_root={SOURCE_ROOT}")
else:
    print("source_root=not_found (will rely on an already importable knee package)")

try:
    from knee.kaggle_runtime import detect_kaggle_paths, KaggleLayout
    from knee.lm_format_enforcer_wheels import (
        install_from_local_wheels,
        runtime_dependency_versions,
        wheels_dir,
    )
    from knee.second_reader_constrained import ensure_lmfe_transformers_compat
    from knee.second_reader_phase47 import KAGGLE_PHASE47_OUTPUT_DIRNAME
except ImportError as exc:
    raise SystemExit(
        "PILOT STOPPED: could not import knee. Attach the private source dataset. "
        f"Original error: {exc}"
    ) from exc

print("internet_used=False gpu_required=True training_run=False process_report_only=False gold_only=True phase47=True")
print("gold_rerun_authorized=false gold_rerun_executed=false notebook_07_executed=false")
print("do_not_overwrite=mahdimzafari/rsna-knee-second-reader-gold-pilot v7")
print("do_not_overwrite=mahdimzafari/rsna-knee-second-reader-constrained-canary v3")
print(detect_kaggle_paths(KaggleLayout(output_dirname=KAGGLE_PHASE47_OUTPUT_DIRNAME)))
wheel_root = wheels_dir(SOURCE_ROOT) if SOURCE_ROOT is not None else wheels_dir()
install_payload = install_from_local_wheels(wheel_root, root=SOURCE_ROOT)
print("lmfe_install=" + json.dumps({
    k: install_payload.get(k)
    for k in ("ok", "internet_used", "version", "license", "installed", "imported_versions")
}, sort_keys=True, default=str))
print("pip_check=" + json.dumps(install_payload.get("pip_check") or {}, sort_keys=True, default=str)[:4000])
print("runtime_versions=" + json.dumps(install_payload.get("runtime_versions") or runtime_dependency_versions(), sort_keys=True, default=str))
if not install_payload.get("ok"):
    raise SystemExit("WHEEL_INSTALL_FAIL")
compat = ensure_lmfe_transformers_compat()
print("lmfe_transformers_compat=" + json.dumps(compat, sort_keys=True))


In [ ]:
# Gate 1 then gold-only v1.4 evaluation. Do not print report text or identifiers.
from pathlib import Path
import json
import time

from knee.kaggle_runtime import KaggleLayout, preflight
from knee.report_labels import (
    HARD_WEAK_LABEL_TRAINING_ALLOWED,
    SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED,
    split_gold_and_report_only,
)
from knee.report_second_reader import inspect_generate_kwargs, locate_weights_directory
from knee.second_reader_json_stop import inspect_termination_from_snapshot
from knee.second_reader_phase46 import FakeSchemaMaskProcessor, gate1_file_hashes, git_head_sha, prove_processor_order
from knee.second_reader_phase47 import (
    EXPECTED_GOLD_STUDIES,
    PHASE47_KERNEL_SLUG,
    count_gold_and_report_only,
    run_phase47_gold_constrained,
)
from knee.second_reader_pilot import load_pilot_config
from knee.second_reader_intervention_audit import semantic_gate_audit
import pandas as pd

assert HARD_WEAK_LABEL_TRAINING_ALLOWED is False
assert SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED is False
print("kernel_slug=" + PHASE47_KERNEL_SLUG)
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-gold-pilot")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-constrained-canary")
print("gold_rerun_authorized=false")
print("process_report_only=false")
print("notebook_07_executed=false")
print("forbidden_outcome=CONSTRAINED_FORMAT_PASS_SEMANTIC_PASS")
print("git_head_sha=" + str(git_head_sha()))
print("yaml_prefix_allowed_tokens_fn=true effective_runtime_prefix_allowed_tokens_fn=false authority=source_code")

cfg_v14 = None
if SOURCE_ROOT is not None:
    cfg_v14 = str(Path(SOURCE_ROOT) / "configs" / "second_reader_pilot_v1_4.yaml")
WEIGHTS = os.environ.get("SECOND_READER_WEIGHTS")
layout = KaggleLayout(output_dirname="rsna_second_reader_gold_constrained_v14")
DATA_ROOT = Path(os.environ["RSNA_DATA_ROOT"]) if os.environ.get("RSNA_DATA_ROOT") else None
OUT_DIR = Path(os.environ["RSNA_SECOND_READER_OUTPUT_DIR"]) if os.environ.get("RSNA_SECOND_READER_OUTPUT_DIR") else Path("/kaggle/working/rsna_second_reader_gold_constrained_v14")
pre = preflight(layout=layout, data_root=DATA_ROOT, output_dir=OUT_DIR)
print(pre.message)
print("data_source=", pre.data_source)
print("scientific_evaluation=", pre.scientific_evaluation)
if not pre.ok or pre.data_root is None:
    raise SystemExit("DATA_NOT_AVAILABLE: attach rsna-knee-abnormality-detection")
train = pd.read_csv(Path(pre.data_root) / "train.csv")
counts = count_gold_and_report_only(train)
print("gold_selection=" + json.dumps(counts, sort_keys=True))
assert counts["n_gold_studies_selected"] == EXPECTED_GOLD_STUDIES
assert counts["n_report_only_studies_selected"] == 0
print("HARD_WEAK_LABEL_TRAINING_ALLOWED=false")
print("SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED=false")

config = load_pilot_config(cfg_v14) if cfg_v14 else {}
assert config.get("process_report_only_studies") is False
weights_dir = locate_weights_directory(config.get("second_reader") or config, WEIGHTS)
print("weights_attached=" + str(weights_dir is not None))
if weights_dir is not None:
    inspection = inspect_termination_from_snapshot(weights_dir)
    print("termination_inspection=" + json.dumps(inspection, sort_keys=True, default=str)[:8000])

hashes = gate1_file_hashes(weights_dir=weights_dir)
print("gate1_file_hashes=" + json.dumps({k: hashes[k] for k in hashes if k != "wheel_manifest"}, sort_keys=True, default=str)[:8000])
if not hashes.get("ok"):
    raise SystemExit("GATE1_FAIL: file hash mismatch")
order = prove_processor_order()
print("processor_order=" + json.dumps(order, sort_keys=True))
if not order.get("ok"):
    raise SystemExit("GATE1_FAIL: processor-order proof")
un_kwargs = {"max_new_tokens": 2048, "do_sample": False, "eos_token_id": [32000, 32007, 32001]}
con_kwargs = dict(un_kwargs)
con_kwargs["logits_processor"] = [FakeSchemaMaskProcessor()]
print("inspect_unconstrained=" + json.dumps(inspect_generate_kwargs(un_kwargs), sort_keys=True))
print("inspect_constrained=" + json.dumps(inspect_generate_kwargs(con_kwargs), sort_keys=True))
print("semantic_gate=" + json.dumps(semantic_gate_audit(), sort_keys=True))
try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    print("gpu_name=" + gpu_name)
    if gpu_name == "cpu":
        raise SystemExit("GATE1_FAIL: cuda_required")
except Exception as exc:
    raise SystemExit("GATE1_FAIL: cuda_required:" + str(exc))

started = time.time()
result = run_phase47_gold_constrained(
    layout=layout,
    data_root=DATA_ROOT if DATA_ROOT is not None else pre.data_root,
    output_dir=OUT_DIR,
    data_source=None if DATA_ROOT is None else os.environ.get("RSNA_REPORT_LABEL_DATA_SOURCE"),
    config_path=cfg_v14,
    weights_path=weights_dir,
)
elapsed = round(time.time() - started, 3)
peak_gpu = None
try:
    import torch
    if torch.cuda.is_available():
        peak_gpu = int(torch.cuda.max_memory_allocated())
except Exception:
    peak_gpu = None
public = {
    "status": result.get("status"),
    "ok": result.get("ok"),
    "skip_reason": result.get("skip_reason"),
    "gold_eval_executed": result.get("gold_eval_executed"),
    "scientific_evaluation": result.get("scientific_evaluation"),
    "execution_status": result.get("execution_status"),
    "semantic_gate_status": result.get("semantic_gate_status"),
    "gold_rerun_authorized": result.get("gold_rerun_authorized"),
    "gold_rerun_executed": result.get("gold_rerun_executed"),
    "notebook_07_executed": result.get("notebook_07_executed"),
    "n_gold_studies_processed": result.get("n_gold_studies_processed"),
    "n_valid_second_reader_outputs": result.get("n_valid_second_reader_outputs"),
    "n_report_only_studies_processed": result.get("n_report_only_studies_processed"),
    "malformed_output_rate": result.get("malformed_output_rate"),
    "kernel_slug": result.get("kernel_slug"),
    "inference_seconds": result.get("wall_seconds") or elapsed,
    "peak_gpu_memory_bytes": result.get("peak_gpu_memory_bytes") or peak_gpu,
    "gate1_ok": (result.get("gate1") or {}).get("ok"),
    "yaml_prefix_allowed_tokens_fn": result.get("yaml_prefix_allowed_tokens_fn"),
    "effective_runtime_prefix_allowed_tokens_fn": result.get("effective_runtime_prefix_allowed_tokens_fn"),
    "code_never_coerces_unknown_to_negative": result.get("code_never_coerces_unknown_to_negative"),
    "forbidden_outcome_not_used": result.get("forbidden_outcome_not_used"),
    "HARD_WEAK_LABEL_TRAINING_ALLOWED": result.get("HARD_WEAK_LABEL_TRAINING_ALLOWED"),
    "SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED": result.get("SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED"),
}
print("phase47=" + json.dumps(public, sort_keys=True, default=str)[:12000])
print("gold_rerun_authorized=false gold_rerun_executed=false notebook_07_executed=false")
if result.get("status") == "CONSTRAINED_FORMAT_PASS_SEMANTIC_PASS":
    raise SystemExit("FORBIDDEN_OUTCOME: CONSTRAINED_FORMAT_PASS_SEMANTIC_PASS")
if result.get("n_gold_studies_processed") != 58:
    raise SystemExit("expected n_gold_studies_processed=58")
if result.get("n_report_only_studies_processed") != 0:
    raise SystemExit("report-only studies were processed")
assert result.get("training_run") is False
assert result.get("folds_created") is False
assert result.get("download_weights") is False
assert result.get("internet_used") is False
assert result.get("gold_rerun_authorized") is False


In [ ]:
# Display sanitized aggregates only. Never read or print private row-level files.
from pathlib import Path
import json

out = Path(result["output_dir"])
sanitized = out / "sanitized"
print("sanitized_dir=", sanitized)
print("DATA_SOURCE=", result.get("data_source"))
print("scientific_evaluation=", result.get("scientific_evaluation"))
print("execution_status=", result.get("execution_status"))
print("n_gold_studies_processed=", result.get("n_gold_studies_processed"))
print("n_report_only_studies_processed=", result.get("n_report_only_studies_processed"))
print("malformed_output_rate=", result.get("malformed_output_rate"))
print("private hashes recorded:", sorted((result.get("private_hashes") or {}).keys()))
print("acceptance=" + json.dumps(result.get("acceptance") or {}, sort_keys=True, default=str)[:8000])
print("recorder=" + json.dumps(result.get("recorder") or {}, sort_keys=True, default=str)[:8000])
print("macro=" + json.dumps(result.get("macro") or {}, sort_keys=True, default=str)[:4000])

summary_path = sanitized / "second_reader_gold_pilot_summary.md"
if summary_path.is_file():
    print(summary_path.read_text(encoding="utf-8")[:12000])

for name in (
    "second_reader_phase47_gold.json",
    "freeze_manifest.json",
    "runtime_metrics.json",
    "pilot_acceptance.json",
    "private_output_hashes.json",
    "generation_diagnostics.json",
):
    path = sanitized / name
    if path.is_file():
        print(f"\n=== {name} ===")
        print(path.read_text(encoding="utf-8")[:4000])

print("gold_rerun_authorized=false")
print("HARD_WEAK_LABEL_TRAINING_ALLOWED=false")
print("SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED=false")
print("notebook_07_executed=false")
